# A2 · Gradient boosting for classification: gradient, hessian and Newton step

**Type**: Implement from scratch · **Level**: ●●● challenge · **Time**: about 60 minutes · **extended path**

## Objective

Implement boosting with the logistic loss: initial prediction as log-odds, **gradient** ($y - p$) as the target of the tree and leaf values with the **Newton step** ($\sum r / \sum p(1-p)$, that is gradient divided by hessian). Check that it matches `GradientBoostingClassifier` in the first rounds and measure how much
a plain gradient step (without the Newton factor) departs.

## Dataset

`make_classification(1500, 8 features, 4 informative, flip_y=0.05)`, 60/40 split: `X_tr, y_tr, X_te, y_te`.

## Educational purpose

Understand where the math of XGBoost comes from: it uses the gradient **and the hessian** of the loss (a second-order expansion). The optimal value of a leaf is the ratio between the sum of the gradients and the sum of the hessians: it is the Newton step for the problem of that leaf.

## Task

1. `sigmoid(z)`. `fit_gbc(X, y, n_rounds, eta, depth)`: `F0 = log(p0 / (1 − p0))` with `p0 = y.mean()`; for each round: `p = sigmoid(F)`, residuals `r = y − p`, tree `DecisionTreeRegressor(max_depth=depth, criterion="squared_error", random_state=0)` trained on `r`; for each **leaf** (`tree.apply(X)`) the value `Σ r / Σ p(1−p)` (over the examples of the leaf) and update `F += eta * leaf_value`. Return `(F0, [(tree, leaf→value dictionary)])`.
2. `predict_proba(model, X, eta)`: probability of class 1 (start from `F0`, add `eta * value` for each tree, apply the sigmoid; leaves not seen in training: it does not happen, the structure is the same).
3. Compare with `GradientBoostingClassifier(n_estimators=B, learning_rate=0.1, max_depth=3, criterion="squared_error", random_state=0)`: save `diff_3` and `diff_50` (maximum absolute difference between the test probabilities after 3 and 50 rounds) and `mean_diff_50` (mean absolute difference after 50 rounds). Save `class_agreement_50` (fraction of test examples with the same predicted class).
4. **Without the Newton factor:** repeat with leaf value = **mean of the residuals** (gradient only, without dividing by the hessian) and save `acc_without_newton` and `acc_with_newton` at 50 rounds with `eta=0.1`; and `logloss_without_newton`, `logloss_with_newton`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

X, y = make_classification(1500, n_features=8, n_informative=4, class_sep=0.8, flip_y=0.05, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)

## Your work

In [ ]:
def sigmoid(z):
    ...


def fit_gbc(X, y, n_rounds, eta, depth, newton=True):
    ...


def predict_proba(model, X, eta):
    ...

In [ ]:
diff_3 = None
diff_50 = None
mean_diff_50 = None
class_agreement_50 = None

In [ ]:
acc_without_newton = None
acc_with_newton = None
logloss_without_newton = None
logloss_with_newton = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Leaf values: `leaves = tree.apply(X)`; `{l: r[leaves == l].sum() / (p[leaves == l] * (1 - p[leaves == l])).sum() for l in np.unique(leaves)}`; update: `F += eta * np.array([values[l] for l in leaves])`. Without Newton: `r[leaves == l].mean()`. Prediction: for a new example `tree.apply(X)` returns the leaf.'):
    check(close(sigmoid(np.array([0.0, 30.0, -30.0])), [0.5, 1.0, 0.0], atol=1e-9), "correct sigmoid", "1 / (1 + exp(-z))")
    m = fit_gbc(X_tr, y_tr, 1, 0.1, 3)
    check(len(m[1]) == 1 and close(m[0], np.log(y_tr.mean() / (1 - y_tr.mean()))), "a single round, F0 = log-odds of the prevalence", "F0 = log(p0 / (1 - p0))")
    check(diff_3 < 1e-9, "after 3 rounds the probabilities match scikit-learn (1e-9)", "leaf values: sum of the residuals / sum of p (1 - p)")
    check(mean_diff_50 < 0.005 and class_agreement_50 > 0.99, "after 50 rounds the mean difference is tiny (below 0.005) and the classes match for over 99%", "differences from float32 and tie-breaking")
    check(acc_without_newton > 0.7 and abs(acc_with_newton - acc_without_newton) < 0.05 and logloss_without_newton > logloss_with_newton + 0.05, "without the Newton factor the log-loss is worse by over 0.05 (the steps are too small), the accuracy almost the same", "compare the two versions")
done()

---
The commented solution is in `solutions/A_from_scratch/A2_classification_gradient_boosting_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.